In [ ]:
import numpy as np
from estploidy.fit_mixtures.gmm import fit_gmm_to_ab
from estploidy.fit_mixtures.lmm import fit_mixed_model_ab

temp_dir='test/temp'
ploidy_results = []
ab_left = np.random.normal(0.25, 0.05, (500, 4))
ab_middle = np.random.normal(0.5, 0.05, (1000, 4))
ab_right = np.random.normal(0.75, 0.05, (500,4))
allele_balance_array = np.concatenate([ab_left, ab_middle, ab_right], axis=0)
#print(allele_balance_array)
allele_depth_array = np.random.randint(10, 100, size=(2000, 4))
allele_mask_array = np.random.randint(0, 2, size=(2000, 4))
ab_dat = np.array([allele_balance_array,allele_depth_array,allele_mask_array])
#print(ab_dat)

for i in range(len(ab_dat[0,0,:])):
    print(i)
    ind_dat = ab_dat[0,:,i]
    depth_dat = ab_dat[1,:,i]
    #print(ind_dat)
    ind_mask = (ab_dat[2,:,i] == 1)
    #print(ind_mask)
    ind_dat_filtered = ind_dat[ind_mask]
    depth_dat_filtered = depth_dat[ind_mask]
    dat = ind_dat_filtered.reshape(-1,1)
    best_n, predictions = fit_gmm_to_ab(ind_name = f'{i}', dat = dat, ploidy = [2,3,4,5,6], model_constraints = 1, output_dir = temp_dir)
    lmm_output = fit_mixed_model_ab(i, ind_dat_filtered, depth_dat_filtered, predictions, temp_dir)
    if lmm_output and len(lmm_output) == 3:
        lmm_result, rand_effects, fixed_effects = lmm_output
        print(lmm_result.summary())
    else:
        print(f"fit_mixed_model_ab did not return 3 values for i={i}, got: {lmm_output}")
    ploidy_results.append(best_n)
print(ploidy_results)
#assert ploidy_results == [4,4,4,4]





In [ ]:
import numpy as np
from estploidy.fit_mixtures.gmm import fit_gmm_to_ab
from estploidy.fit_mixtures.lmm import fit_mixed_model_ab

temp_dir='test/temp'

np.random.seed(3232)
ab_left = np.random.normal(0.25, 0.05, (500, 4))
ab_middle = np.random.normal(0.5, 0.05, (1000, 4))
ab_right = np.random.normal(0.75, 0.05, (500,4))
allele_balance_array = np.concatenate([ab_left, ab_middle, ab_right], axis=0)
allele_depth_array = np.random.randint(20, 100, size=(2000, 4))
allele_mask_array = np.random.randint(0, 2, size=(2000, 4))
ab_dat = np.array([allele_balance_array,allele_depth_array,allele_mask_array])
for i in range(len(ab_dat[0,0,:])):
    #print(i)
    ind_dat = ab_dat[0,:,i]
    #print(ind_dat)
    ind_depth = ab_dat[1,:,i]
    ind_mask = (ab_dat[2,:,i] == 1)
    #print(ind_mask)
    ind_dat_filtered = ind_dat[ind_mask]
    ind_depth_filtered = ind_depth[ind_mask]
    dat = ind_dat_filtered.reshape(-1,1)
    best_n, predictions = fit_gmm_to_ab(ind_name = f'{i}', dat = dat, ploidy = [2,3,4,5,6], model_constraints = 1, output_dir = temp_dir)
    #print(best_n)
    #print(predictions)
    lmm_result, rand_effects, fixed_effects, p_value = fit_mixed_model_ab(ind_name = f'{i}', allele_balance_data = ind_dat_filtered, site_depth_data = ind_depth_filtered, gmm_predictions = predictions, output_dir = temp_dir)
    print(rand_effects)
    print(fixed_effects)
    print(f'Checking Random Effects for dataset {i}')
    for group, effect in lmm_result.random_effects.items():
        print(group)
        print(effect)
        group_effect = effect.iloc[0]
        print(group_effect)
        print('Done Checking Effects!\n')